# graph-vul: Fusion Method AblationThis notebook compares **8 different fusion mechanisms** for injecting graph(AST+CFG) information into UniXcoder transformer layers.All experiments use all 12 encoder layers with late graph concat (except E0 baseline).The only variable is *how* graph information is injected.**Experiments:**| Tag | Method | Description ||-----|--------|-------------|| E0 | Late only | No internal fusion, late concat baseline || E1 | Gated residual | Current default: `hidden + sigmoid(gate) * proj(graph)` || E2 | Ungated residual | Simple additive: `hidden + proj(graph)` || E3 | Scalar-gated residual | Single learned scalar gate || E4 | FiLM | Feature-wise Linear Modulation: `gamma * hidden + beta` || E5 | Concat adapter | Concat + bottleneck + LayerNorm + residual || E6 | Gated concat adapter | Gated version of concat adapter || E7 | Cross-attention | Cross-attn from code tokens to K=8 learned graph tokens |**Decision rule:** best unseen 0–64 F1 first, unseen overall F1 second, simplicity third.

In [1]:
import subprocess

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU."
    )
print(gpu_info.stdout)

Sun Mar 29 22:28:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/data/graph-vul"
DRIVE_DATA = f"{DRIVE_PROJECT}/data"
DRIVE_HF_CACHE = f"{DRIVE_PROJECT}/hf_cache"

Mounted at /content/drive


In [3]:
!mkdir -p {DRIVE_DATA}
!mkdir -p {DRIVE_HF_CACHE}

In [4]:
import csv
from pathlib import Path

data_root = Path(DRIVE_DATA)
subset_path = data_root / "func_subset.csv"
assignments_path = data_root / "func_splits" / "assignments.csv"
embeddings_dir = data_root / "processed" / "cwe_embeddings"

required_files = [subset_path, assignments_path, embeddings_dir]
missing = [str(path.relative_to(data_root)) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required paths in {DRIVE_DATA}:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )

required_splits = {"train", "val", "test", "unseen_val", "unseen_test"}
with assignments_path.open(newline="") as handle:
    reader = csv.DictReader(handle)
    observed_splits = {row["split"] for row in reader if row.get("split")}

missing_splits = sorted(required_splits - observed_splits)
if missing_splits:
    raise ValueError(
        "assignments.csv is missing required splits:\n"
        + "\n".join(f"  - {split}" for split in missing_splits)
    )

with subset_path.open(newline="") as handle:
    reader = csv.DictReader(handle)
    cwe_ids = sorted({row["cwe_id"].strip() for row in reader if row.get("cwe_id", "").strip()})

missing_embeddings = [cwe_id for cwe_id in cwe_ids if not (embeddings_dir / f"{cwe_id}.pt").exists()]
if missing_embeddings:
    raise FileNotFoundError(
        "Missing CWE embeddings for func_subset.csv entries:\n"
        + "\n".join(f"  - processed/cwe_embeddings/{cwe_id}.pt" for cwe_id in missing_embeddings)
    )

print(f"Validated dataset root: {data_root}")
print(f"Found {len(cwe_ids)} CWE embedding files referenced by func_subset.csv.")
print(f"Observed splits: {sorted(observed_splits)}")

Validated dataset root: /content/drive/MyDrive/data/graph-vul/data
Found 89 CWE embedding files referenced by func_subset.csv.
Observed splits: ['test', 'train', 'unseen_test', 'unseen_val', 'val']


In [5]:
# Validate graph artifacts (always required for fusion ablation)
graph_dir = data_root / "processed" / "graph_artifacts"
node_vocab = graph_dir / "node_type_vocab.json"
edge_vocab = graph_dir / "edge_type_vocab.json"

missing_graph = []
if not graph_dir.exists():
    missing_graph.append(str(graph_dir.relative_to(data_root)))
if not node_vocab.exists():
    missing_graph.append(str(node_vocab.relative_to(data_root)))
if not edge_vocab.exists():
    missing_graph.append(str(edge_vocab.relative_to(data_root)))
if missing_graph:
    raise FileNotFoundError(
        f"Graph artifacts required for fusion ablation but missing in {DRIVE_DATA}:\n"
        + "\n".join(f"  - {p}" for p in missing_graph)
    )

graph_pts = list(graph_dir.glob("*.pt"))
print(f"Graph artifacts validated: {len(graph_pts)} .pt files found in {graph_dir}")

Graph artifacts validated: 1733 .pt files found in /content/drive/MyDrive/data/graph-vul/data/processed/graph_artifacts


In [6]:
import os

os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_HF_CACHE

In [7]:
!git clone https://github.com/noamdwc/graph-vul.git /content/graph-vul
%pip install -q pytorch-lightning transformers sentencepiece pandas numpy \
    scikit-learn torchmetrics wandb torch-geometric

Cloning into '/content/graph-vul'...
remote: Enumerating objects: 796, done.
remote: Counting objects: 100% (796/796), done.
remote: Compressing objects: 100% (444/444), done.
remote: Total 796 (delta 333), reused 677 (delta 216), pack-reused 0 (from 0)
Receiving objects: 100% (796/796), 2.35 MiB | 7.73 MiB/s, done.
Resolving deltas: 100% (333/333), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.6 MB/s eta 0:00:00


In [8]:
import os
import shutil
from pathlib import Path

repo_data = "/content/graph-vul/data"

if os.path.islink(repo_data):
    os.unlink(repo_data)
elif os.path.isdir(repo_data):
    shutil.rmtree(repo_data)

os.symlink(DRIVE_DATA, repo_data)
assert Path(repo_data, "func_subset.csv").exists(), "Symlink verification failed"
print(f"Symlinked {repo_data} -> {DRIVE_DATA}")

Symlinked /content/graph-vul/data -> /content/drive/MyDrive/data/graph-vul/data


## Weights & Biases

The training script logs to WandB. Run the cell below to authenticate.
To skip WandB logging, uncomment the `WANDB_MODE` line instead.

In [9]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: the-doctor (the-doctor-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Fusion Module DefinitionsAll 7 fusion modules defined below, plus a registry dict. Each moduleimplements `forward(hidden_state, graph_emb) -> fused_hidden_state` (exceptcross-attention which takes graph tokens instead of a pooled vector).

In [ ]:
import torchimport torch.nn as nnfrom torch_geometric.nn import global_mean_poolclass GatedResidualFusion(nn.Module):    """hidden + sigmoid(gate(hidden)) * proj(graph). Gate bias=-2.0."""    def __init__(self, graph_dim: int, hidden_size: int) -> None:        super().__init__()        self.proj = nn.Linear(graph_dim, hidden_size)        self.gate = nn.Linear(hidden_size, 1)        nn.init.constant_(self.gate.bias, -2.0)    def forward(self, hidden_state: torch.Tensor, graph_emb: torch.Tensor) -> torch.Tensor:        projected = self.proj(graph_emb).unsqueeze(1)        gate_val = torch.sigmoid(self.gate(hidden_state))        return hidden_state + gate_val * projectedclass UngatedResidualFusion(nn.Module):    """hidden + proj(graph). No gating."""    def __init__(self, graph_dim: int, hidden_size: int) -> None:        super().__init__()        self.proj = nn.Linear(graph_dim, hidden_size)    def forward(self, hidden_state: torch.Tensor, graph_emb: torch.Tensor) -> torch.Tensor:        projected = self.proj(graph_emb).unsqueeze(1)        return hidden_state + projectedclass ScalarGatedResidualFusion(nn.Module):    """hidden + sigmoid(alpha) * proj(graph). Alpha is a single scalar, init=-2.0."""    def __init__(self, graph_dim: int, hidden_size: int) -> None:        super().__init__()        self.proj = nn.Linear(graph_dim, hidden_size)        self.alpha = nn.Parameter(torch.tensor(-2.0))    def forward(self, hidden_state: torch.Tensor, graph_emb: torch.Tensor) -> torch.Tensor:        projected = self.proj(graph_emb).unsqueeze(1)        return hidden_state + torch.sigmoid(self.alpha) * projectedclass FiLMFusion(nn.Module):    """gamma(graph) * hidden + beta(graph). Identity init: gamma_bias=1, beta_bias=0."""    def __init__(self, graph_dim: int, hidden_size: int) -> None:        super().__init__()        self.gamma = nn.Linear(graph_dim, hidden_size)        self.beta = nn.Linear(graph_dim, hidden_size)        nn.init.constant_(self.gamma.bias, 1.0)        nn.init.zeros_(self.gamma.weight)        nn.init.zeros_(self.beta.bias)        nn.init.zeros_(self.beta.weight)    def forward(self, hidden_state: torch.Tensor, graph_emb: torch.Tensor) -> torch.Tensor:        gamma = self.gamma(graph_emb).unsqueeze(1)        beta = self.beta(graph_emb).unsqueeze(1)        return gamma * hidden_state + betaclass ConcatAdapterFusion(nn.Module):    """Concat hidden with proj(graph), bottleneck back to hidden_size, residual."""    def __init__(self, graph_dim: int, hidden_size: int) -> None:        super().__init__()        self.proj = nn.Linear(graph_dim, hidden_size)        self.down = nn.Linear(hidden_size * 2, hidden_size)        self.ln = nn.LayerNorm(hidden_size)    def forward(self, hidden_state: torch.Tensor, graph_emb: torch.Tensor) -> torch.Tensor:        projected = self.proj(graph_emb).unsqueeze(1).expand_as(hidden_state)        concat = torch.cat([hidden_state, projected], dim=-1)        adapter_out = self.ln(torch.relu(self.down(concat)))        return hidden_state + adapter_outclass GatedConcatAdapterFusion(nn.Module):    """Gated concat adapter: hidden + sigmoid(gate) * adapter_output. Gate bias=-2.0."""    def __init__(self, graph_dim: int, hidden_size: int) -> None:        super().__init__()        self.proj = nn.Linear(graph_dim, hidden_size)        self.down = nn.Linear(hidden_size * 2, hidden_size)        self.ln = nn.LayerNorm(hidden_size)        self.gate = nn.Linear(hidden_size, 1)        nn.init.constant_(self.gate.bias, -2.0)    def forward(self, hidden_state: torch.Tensor, graph_emb: torch.Tensor) -> torch.Tensor:        projected = self.proj(graph_emb).unsqueeze(1).expand_as(hidden_state)        concat = torch.cat([hidden_state, projected], dim=-1)        adapter_out = self.ln(torch.relu(self.down(concat)))        gate_val = torch.sigmoid(self.gate(hidden_state))        return hidden_state + gate_val * adapter_outclass GraphTokenPooling(nn.Module):    """Attention-pool variable-length node embeddings into K fixed graph tokens."""    def __init__(self, graph_dim: int, num_tokens: int = 8) -> None:        super().__init__()        self.num_tokens = num_tokens        self.queries = nn.Parameter(torch.randn(num_tokens, graph_dim) * 0.02)        self.attn = nn.MultiheadAttention(embed_dim=graph_dim, num_heads=4, batch_first=True)    def forward(        self,        node_embeddings: torch.Tensor,        batch_index: torch.Tensor,        num_graphs: int,    ) -> torch.Tensor:        """        Args:            node_embeddings: [N_total, graph_dim] all nodes across batch            batch_index: [N_total] graph index per node            num_graphs: batch size        Returns:            [B, K, graph_dim] graph tokens        """        device = node_embeddings.device        graph_dim = node_embeddings.shape[-1]        K = self.num_tokens        if node_embeddings.shape[0] == 0:            return torch.zeros(num_graphs, K, graph_dim, device=device)        # Pad per-graph node sets into [B, max_nodes, graph_dim]        counts = torch.zeros(num_graphs, dtype=torch.long, device=device)        counts.scatter_add_(0, batch_index, torch.ones_like(batch_index))        max_nodes = counts.max().item()        padded = torch.zeros(num_graphs, max_nodes, graph_dim, device=device)        key_padding_mask = torch.ones(num_graphs, max_nodes, dtype=torch.bool, device=device)        offsets = torch.zeros(num_graphs, dtype=torch.long, device=device)        offsets[1:] = counts[:-1].cumsum(0)        for g in range(num_graphs):            n = counts[g].item()            if n > 0:                padded[g, :n] = node_embeddings[offsets[g]:offsets[g] + n]                key_padding_mask[g, :n] = False        queries = self.queries.unsqueeze(0).expand(num_graphs, -1, -1)        tokens, _ = self.attn(queries, padded, padded, key_padding_mask=key_padding_mask)        return tokensclass CrossAttentionGraphTokensFusion(nn.Module):    """Cross-attention: code tokens attend to K graph tokens, then residual."""    def __init__(self, graph_dim: int, hidden_size: int, num_heads: int = 8) -> None:        super().__init__()        self.kv_proj = nn.Linear(graph_dim, hidden_size)        self.cross_attn = nn.MultiheadAttention(            embed_dim=hidden_size, num_heads=num_heads, batch_first=True,        )        self.ln = nn.LayerNorm(hidden_size)    def forward(self, hidden_state: torch.Tensor, graph_tokens: torch.Tensor) -> torch.Tensor:        """        Args:            hidden_state: [B, seq_len, hidden_size]            graph_tokens: [B, K, graph_dim]        Returns:            [B, seq_len, hidden_size]        """        kv = self.kv_proj(graph_tokens)        attn_out, _ = self.cross_attn(hidden_state, kv, kv)        return self.ln(hidden_state + attn_out)FUSION_REGISTRY = {    "gated_residual": GatedResidualFusion,    "ungated_residual": UngatedResidualFusion,    "scalar_gated_residual": ScalarGatedResidualFusion,    "film": FiLMFusion,    "concat_adapter": ConcatAdapterFusion,    "gated_concat_adapter": GatedConcatAdapterFusion,    "cross_attention_graph_tokens": CrossAttentionGraphTokensFusion,}print(f"Registered {len(FUSION_REGISTRY)} fusion methods: {list(FUSION_REGISTRY.keys())}")

## Model PatchingMonkey-patch `FuncFinetuneModel` to accept a `fusion_method` string parameterand instantiate the corresponding module from `FUSION_REGISTRY`.For E7 (cross-attention), we also patch the graph encoder to return per-nodeembeddings and add a `GraphTokenPooling` wrapper.

In [ ]:
import syssys.path.insert(0, "/content/graph-vul")import torchimport torch.nn as nnfrom torch_geometric.nn import global_mean_poolfrom graph_vul.model.func_finetune_model import FuncFinetuneModelfrom graph_vul.model.graph_encoder import FunctionGraphEncoder# Store originals_orig_init = FuncFinetuneModel.__init___orig_register_hooks = FuncFinetuneModel._register_fusion_hooks_orig_forward = FuncFinetuneModel.forward_orig_graph_forward = FunctionGraphEncoder.forwarddef _patched_graph_forward(self, node_type_ids, edge_index, edge_type_ids, batch, num_graphs=None):    """Return (pooled, node_embeddings) when _return_node_embeddings is set."""    if num_graphs is None:        num_graphs = batch.max().item() + 1 if batch.numel() > 0 else 1    if node_type_ids.shape[0] == 0:        pooled = torch.zeros(num_graphs, self.hidden_dim, device=node_type_ids.device)        if getattr(self, '_return_node_embeddings', False):            return pooled, torch.zeros(0, self.hidden_dim, device=node_type_ids.device), batch        return pooled    x = self.node_embedding(node_type_ids)    for conv in self.convs:        x = conv(x, edge_index, edge_type_ids)        x = torch.relu(x)        x = self.dropout(x)    pooled = global_mean_pool(x, batch, size=num_graphs)    if getattr(self, '_return_node_embeddings', False):        return pooled, x, batch    return pooleddef _patched_init(self, *args, fusion_method=None, num_graph_tokens=8, **kwargs):    """Accept fusion_method string and look up class from FUSION_REGISTRY."""    # Temporarily remove fusion_layers to prevent original init from creating GatedResidualFusion    orig_fusion_layers = kwargs.get('fusion_layers', None)    if fusion_method is not None and fusion_method != 'gated_residual':        kwargs['fusion_layers'] = None    _orig_init(self, *args, **kwargs)    # Now set up fusion with the chosen method    if fusion_method is not None and orig_fusion_layers and self.graph_dim > 0:        self.fusion_layers = orig_fusion_layers        self._fusion_method = fusion_method        self._num_graph_tokens = num_graph_tokens        encoder_dim = self.encoder.config.hidden_size        if fusion_method == 'cross_attention_graph_tokens':            FusionClass = FUSION_REGISTRY[fusion_method]            self.fusion_modules = nn.ModuleDict(                {str(i): FusionClass(self.graph_dim, encoder_dim) for i in orig_fusion_layers}            )            self.graph_token_pooling = GraphTokenPooling(self.graph_dim, num_graph_tokens)            self.graph_encoder.forward = _patched_graph_forward.__get__(self.graph_encoder, FunctionGraphEncoder)            self.graph_encoder._return_node_embeddings = True        elif fusion_method != 'gated_residual':            FusionClass = FUSION_REGISTRY[fusion_method]            self.fusion_modules = nn.ModuleDict(                {str(i): FusionClass(self.graph_dim, encoder_dim) for i in orig_fusion_layers}            )        self._register_fusion_hooks()    elif fusion_method is None or fusion_method == 'gated_residual':        self._fusion_method = fusion_method or 'gated_residual'def _patched_register_hooks(self):    """Register hooks — cross-attention gets graph tokens, others get pooled vector."""    for handle in self._fusion_hooks:        handle.remove()    self._fusion_hooks.clear()    if not self.fusion_layers:        return    encoder_layers = self.encoder.encoder.layer    is_cross_attn = getattr(self, '_fusion_method', None) == 'cross_attention_graph_tokens'    for layer_idx in self.fusion_layers:        fusion_mod = self.fusion_modules[str(layer_idx)]        if is_cross_attn:            def make_hook(mod):                def hook_fn(layer_module, input, output):                    graph_tokens = getattr(self, '_current_graph_tokens', None)                    if graph_tokens is None:                        return output                    if isinstance(output, tuple):                        hidden_state = output[0]                        fused = mod(hidden_state, graph_tokens)                        return (fused,) + output[1:]                    return mod(output, graph_tokens)                return hook_fn        else:            def make_hook(mod):                def hook_fn(layer_module, input, output):                    graph_emb = self._current_graph_emb                    if graph_emb is None:                        return output                    if isinstance(output, tuple):                        hidden_state = output[0]                        fused = mod(hidden_state, graph_emb)                        return (fused,) + output[1:]                    return mod(output, graph_emb)                return hook_fn        handle = encoder_layers[layer_idx].register_forward_hook(make_hook(fusion_mod))        self._fusion_hooks.append(handle)def _patched_forward(self, input_ids, attention_mask, cwe_embedding, graph_batch=None):    """Forward with cross-attention graph token support."""    graph_emb = None    is_cross_attn = getattr(self, '_fusion_method', None) == 'cross_attention_graph_tokens'    if self.graph_dim > 0 and graph_batch is not None:        if is_cross_attn:            result = self.graph_encoder(                graph_batch.node_type_ids, graph_batch.edge_index,                graph_batch.edge_type_ids, graph_batch.batch,                num_graphs=graph_batch.num_graphs,            )            graph_emb, node_embs, batch_idx = result            self._current_graph_tokens = self.graph_token_pooling(                node_embs, batch_idx, graph_batch.num_graphs,            )        else:            graph_emb = self.graph_encoder(                graph_batch.node_type_ids, graph_batch.edge_index,                graph_batch.edge_type_ids, graph_batch.batch,                num_graphs=graph_batch.num_graphs,            )    if self.fusion_layers and self.graph_dim > 0:        if is_cross_attn:            if not hasattr(self, '_current_graph_tokens') or self._current_graph_tokens is None:                self._current_graph_tokens = None        else:            if graph_emb is not None:                self._current_graph_emb = graph_emb            else:                self._current_graph_emb = torch.zeros(                    input_ids.shape[0], self.graph_dim, device=input_ids.device,                )    outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)    hidden = outputs.last_hidden_state    self._current_graph_emb = None    self._current_graph_tokens = None    mask = attention_mask.unsqueeze(-1).float()    pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)    parts = [pooled]    if self.graph_dim > 0 and self.final_graph_concat and graph_emb is not None:        parts.append(graph_emb)    parts.append(cwe_embedding)    fused = torch.cat(parts, dim=-1)    return self.head(fused)# Apply patchesFuncFinetuneModel.__init__ = _patched_initFuncFinetuneModel._register_fusion_hooks = _patched_register_hooksFuncFinetuneModel.forward = _patched_forwardprint("Patched FuncFinetuneModel with fusion_method support")print("Patched FunctionGraphEncoder with _return_node_embeddings support")

## Ablation ConfigurationShared hyperparameters (identical across all experiments) and the 8-experimentablation suite. Each experiment varies only `fusion_method`.**OOM?** Reduce `BATCH_SIZE` to 4 and increase `GRAD_ACCUM` to 16 to keep effective batch size at 64.

In [ ]:
# Shared hyperparametersNUM_EPOCHS = 10BATCH_SIZE = 64ENCODER_LR = 2e-5HEAD_LR = 1e-3GRAD_ACCUM = 8SEED = 42# Ablation suite — each dict is one experimentEXPERIMENTS = [    {        "run_tag": "E0_late_only",        "fusion_method": None,        "fusion_layers": None,        "final_graph_concat": True,    },    {        "run_tag": "E1_gated_residual",        "fusion_method": "gated_residual",        "fusion_layers": list(range(12)),        "final_graph_concat": True,    },    {        "run_tag": "E2_ungated_residual",        "fusion_method": "ungated_residual",        "fusion_layers": list(range(12)),        "final_graph_concat": True,    },    {        "run_tag": "E3_scalar_gated_residual",        "fusion_method": "scalar_gated_residual",        "fusion_layers": list(range(12)),        "final_graph_concat": True,    },    {        "run_tag": "E4_film",        "fusion_method": "film",        "fusion_layers": list(range(12)),        "final_graph_concat": True,    },    {        "run_tag": "E5_concat_adapter",        "fusion_method": "concat_adapter",        "fusion_layers": list(range(12)),        "final_graph_concat": True,    },    {        "run_tag": "E6_gated_concat_adapter",        "fusion_method": "gated_concat_adapter",        "fusion_layers": list(range(12)),        "final_graph_concat": True,    },    {        "run_tag": "E7_cross_attention_graph_tokens",        "fusion_method": "cross_attention_graph_tokens",        "fusion_layers": list(range(12)),        "final_graph_concat": True,        "num_graph_tokens": 8,    },]print(f"Configured {len(EXPERIMENTS)} experiments:")for exp in EXPERIMENTS:    fm = exp.get("fusion_method") or "none (late only)"    fl = exp.get("fusion_layers")    fl_str = f"{len(fl)} layers" if fl else "none"    extra = f", K={exp['num_graph_tokens']}" if "num_graph_tokens" in exp else ""    print(f"  {exp['run_tag']}: method={fm}, fusion={fl_str}, late={exp['final_graph_concat']}{extra}")

## Run Ablation SuiteThe cell below loops over all experiments, training each model and extractingpredictions + per-bucket metrics. Completed experiments (checkpoint exists) areskipped for Colab resume safety.

In [ ]:
import json
import logging
import sys
import time

import pandas as pd
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
sys.path.insert(0, "/content/graph-vul")

from pathlib import Path

from graph_vul.train_finetune import train
from graph_vul.export_finetune_predictions import export_split_predictions
from graph_vul.model.func_finetune_lit_module import FuncFinetuneLitModule
from graph_vul.analysis.post_experiment_analysis import (
    canonicalize_prediction_frame,
    stratified_metrics,
    DEFAULT_LENGTH_BUCKETS,
)
from transformers import AutoTokenizer
from graph_vul.config import UNIXCODER_MODEL_NAME

ABLATION_ROOT = Path(DRIVE_DATA) / "fusion_method_ablation"
summary_rows = []

# Save original LitModule init
_orig_lit_init = FuncFinetuneLitModule.__init__

for i, exp in enumerate(EXPERIMENTS):
    run_tag = exp["run_tag"]
    print(f"\n{'='*60}")
    print(f"Experiment {i+1}/{len(EXPERIMENTS)}: {run_tag}")
    print(f"{'='*60}")

    exp_dir = ABLATION_ROOT / run_tag
    exp_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = exp_dir / "func_finetune_model.ckpt"
    metrics_path = exp_dir / "func_finetune_metrics.json"
    pred_dir = exp_dir / "predictions"

    # Skip if already completed
    if ckpt_path.exists() and metrics_path.exists():
        print(f"  Checkpoint exists, loading saved metrics...")
        with open(metrics_path) as f:
            metrics = json.load(f)
        train_minutes = -1
    else:
        # Patch LitModule to pass fusion_method to model
        fusion_method = exp.get("fusion_method")
        num_graph_tokens = exp.get("num_graph_tokens", 8)

        def _make_lit_patch(fm, ngt):
            def _patched_lit_init(self_lit, **kwargs):
                # Call original init (creates model without fusion_method)
                _orig_lit_init(self_lit, **kwargs)
                # If fusion_method specified, re-create model with it
                if fm is not None:
                    self_lit.model = FuncFinetuneModel(
                        graph_dim=kwargs.get('graph_dim', 0),
                        num_node_types=kwargs.get('num_node_types', 0),
                        num_relations=kwargs.get('num_relations', 0),
                        fusion_layers=kwargs.get('fusion_layers', None),
                        final_graph_concat=kwargs.get('final_graph_concat', True),
                        fusion_method=fm,
                        num_graph_tokens=ngt,
                    )
            return _patched_lit_init

        FuncFinetuneLitModule.__init__ = _make_lit_patch(fusion_method, num_graph_tokens)

        t0 = time.time()
        train(
            num_epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            encoder_lr=ENCODER_LR,
            head_lr=HEAD_LR,
            grad_accum=GRAD_ACCUM,
            seed=SEED,
            use_graph=True,
            fusion_layers=exp.get("fusion_layers"),
            final_graph_concat=exp.get("final_graph_concat", True),
            checkpoint_path=ckpt_path,
            metrics_path=metrics_path,
            run_tag=run_tag,
        )
        train_minutes = (time.time() - t0) / 60.0
        print(f"  Training completed in {train_minutes:.1f} minutes")

        # Restore original LitModule init
        FuncFinetuneLitModule.__init__ = _orig_lit_init

        with open(metrics_path) as f:
            metrics = json.load(f)

    # Export predictions
    if not (pred_dir / "test_predictions.csv").exists():
        tokenizer = AutoTokenizer.from_pretrained(UNIXCODER_MODEL_NAME)
        model = FuncFinetuneLitModule.load_from_checkpoint(
            str(ckpt_path),
            map_location="cuda" if torch.cuda.is_available() else "cpu",
        )
        model.eval()

        for split in ("test", "unseen_test"):
            export_split_predictions(
                model=model,
                tokenizer=tokenizer,
                split=split,
                output_dir=str(pred_dir),
                batch_size=BATCH_SIZE,
            )

    # Collect summary row
    row = {
        "run_tag": run_tag,
        "fusion_method": exp.get("fusion_method") or "late_only",
        "test_f1": metrics.get("test_f1"),
        "unseen_test_f1": metrics.get("unseen_test_f1"),
        "unseen_test_precision": metrics.get("unseen_test_precision"),
        "unseen_test_recall": metrics.get("unseen_test_recall"),
        "train_minutes": round(train_minutes, 1),
    }

    # Per-bucket metrics from predictions
    for split_prefix, pred_filename in [("", "test"), ("unseen_", "unseen_test")]:
        pred_path = pred_dir / f"{pred_filename}_predictions.csv"
        if pred_path.exists():
            pred_df = pd.read_csv(pred_path)
            pred_df, _ = canonicalize_prediction_frame(pred_df, split_name=pred_filename)
            token_buckets = DEFAULT_LENGTH_BUCKETS["token_length"]
            strat = stratified_metrics(pred_df, "token_length", token_buckets)

            if not strat.empty:
                for _, brow in strat.iterrows():
                    parts = str(brow.name).strip("[]()").split(",")
                    if len(parts) == 2:
                        lo = parts[0].strip().replace(".0", "")
                        hi = parts[1].strip().replace(".0", "").replace("inf", "inf")
                        bucket_key = f"{lo}_{hi}"
                    else:
                        bucket_key = str(brow.name)

                    row[f"{split_prefix}{bucket_key}_f1"] = brow.get("f1")
                    if bucket_key == "0_64":
                        row[f"{split_prefix}0_64_precision"] = brow.get("precision")
                        row[f"{split_prefix}0_64_recall"] = brow.get("recall")

    summary_rows.append(row)
    print(f"  Metrics: test_f1={row.get('test_f1', 'N/A')}, "
          f"unseen_f1={row.get('unseen_test_f1', 'N/A')}, "
          f"unseen_0_64_f1={row.get('unseen_0_64_f1', 'N/A')}")

# Restore init in case it wasn't restored (e.g., error during training)
FuncFinetuneLitModule.__init__ = _orig_lit_init

print(f"\nAll {len(summary_rows)} experiments completed.")

## Results SummaryComparison table sorted by **unseen 0–64 F1** (primary metric).Best value per column highlighted in green.

In [ ]:
import pandas as pdresults_df = pd.DataFrame(summary_rows)# Primary display columnsdisplay_cols = [    "run_tag",    "fusion_method",    "unseen_test_f1",    "unseen_0_64_f1",    "unseen_64_128_f1",    "test_f1",    "0_64_f1",    "train_minutes",]cols = [c for c in display_cols if c in results_df.columns]primary_df = results_df[cols].copy()# Sort by primary metricif "unseen_0_64_f1" in primary_df.columns:    primary_df = primary_df.sort_values("unseen_0_64_f1", ascending=False)primary_df = primary_df.set_index("run_tag")fmt = {c: "{:.4f}" for c in primary_df.columns if primary_df[c].dtype == "float64" and c != "train_minutes"}if "train_minutes" in primary_df.columns:    fmt["train_minutes"] = "{:.1f}"highlight_cols = [c for c in ["unseen_test_f1", "unseen_0_64_f1", "test_f1", "0_64_f1"] if c in primary_df.columns]print("Fusion Method Ablation Results (sorted by unseen 0-64 F1)")print("=" * 90)display(primary_df.style.format(fmt).highlight_max(subset=highlight_cols, color="lightgreen"))

## Unseen Test Bucket DetailFull per-bucket breakdown on unseen_test split, with precision/recall for 0–64 bucket.

In [ ]:
unseen_detail_cols = [    "run_tag",    "unseen_test_f1",    "unseen_test_precision",    "unseen_test_recall",    "unseen_0_64_f1",    "unseen_0_64_precision",    "unseen_0_64_recall",    "unseen_64_128_f1",    "unseen_128_256_f1",    "unseen_256_512_f1",]cols = [c for c in unseen_detail_cols if c in results_df.columns]detail_df = results_df[cols].copy()if "unseen_0_64_f1" in detail_df.columns:    detail_df = detail_df.sort_values("unseen_0_64_f1", ascending=False)detail_df = detail_df.set_index("run_tag")fmt = {c: "{:.4f}" for c in detail_df.columns if detail_df[c].dtype == "float64"}print("Unseen Test Detail (sorted by unseen 0-64 F1)")print("=" * 90)display(detail_df.style.format(fmt).highlight_max(    subset=[c for c in ["unseen_0_64_f1", "unseen_test_f1"] if c in detail_df.columns],    color="lightgreen",))

In [ ]:
# Save full results to Driveoutput_path = ABLATION_ROOT / "fusion_method_ablation_results.csv"results_df.to_csv(output_path, index=False)print(f"Saved full results to {output_path}")